<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/07_ctd_reasoning_distractors_counterfactual_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 - Standalone CTD reasoning diagnostics

This notebook is self-contained. It does not rely on variables, models, or adapters from another notebook.

It builds a compact CTD chemical-gene-disease reasoning dataset, fine-tunes Qwen2.5-0.5B-Instruct with LoRA SFT, and then evaluates clean, distractor, and synthetic counterfactual cases.

The counterfactual cases are synthetic diagnostic tests, not biomedical claims.

In [1]:
!pip -q install -U "transformers>=4.55,<5" "datasets>=3.6,<5" "peft>=0.17,<1" "trl==0.29.1" "accelerate>=1.10,<2" "bitsandbytes>=0.46,<1" "torchao>=0.16,<1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 150.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 49.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import os
import random
import re
import pandas as pd
import torch
from datasets import Dataset
from google.colab import files

import transformers, datasets, peft, trl
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Datasets:', datasets.__version__)
print('PEFT:', peft.__version__)
print('TRL:', trl.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

print('Upload: CTD_chem_gene_ixns.tsv.gz and CTD_curated_genes_diseases.tsv.gz')
files.upload()

CHEM_GENE = '/content/CTD_chem_gene_ixns.tsv.gz'
GENE_DISEASE = '/content/CTD_curated_genes_diseases.tsv.gz'
if not os.path.exists(CHEM_GENE) or not os.path.exists(GENE_DISEASE):
    raise FileNotFoundError('Both CTD files are required.')

PyTorch: 2.11.0+cu128
Transformers: 4.57.6
Datasets: 4.8.5
PEFT: 0.20.0
TRL: 0.29.1
CUDA: True
GPU: NVIDIA L4
VRAM GB: 22.0
Upload: CTD_chem_gene_ixns.tsv.gz and CTD_curated_genes_diseases.tsv.gz


Saving CTD_curated_genes_diseases.tsv.gz to CTD_curated_genes_diseases.tsv.gz
Saving CTD_chem_gene_ixns.tsv.gz to CTD_chem_gene_ixns.tsv.gz


In [3]:
chem_cols = ['ChemicalName','ChemicalID','CasRN','GeneSymbol','GeneID','GeneForms','Organism','OrganismID','Interaction','InteractionActions','PubMedIDs']
gd_cols = ['GeneSymbol','GeneID','DiseaseName','DiseaseID','DirectEvidence','InferenceChemicalName','InferenceChemicalID','OmimIDs','PubMedIDs']

chem = pd.read_csv(CHEM_GENE, sep='\t', comment='#', header=None, names=chem_cols, dtype=str, low_memory=False)
gd = pd.read_csv(GENE_DISEASE, sep='\t', comment='#', header=None, names=gd_cols, dtype=str, low_memory=False)

chem = chem[chem['OrganismID'].fillna('').str.strip().eq('9606')].copy()
chem = chem[chem['ChemicalName'].notna() & chem['GeneSymbol'].notna() & chem['GeneID'].notna()].copy()
gd = gd[gd['GeneID'].notna() & gd['DiseaseName'].notna() & gd['DiseaseID'].notna()].copy()
chem['GeneID'] = chem['GeneID'].str.replace(r'\.0$', '', regex=True)
gd['GeneID'] = gd['GeneID'].str.replace(r'\.0$', '', regex=True)
gd = gd.drop_duplicates(['GeneID','DiseaseID']).copy()

pairs = chem.merge(gd[['GeneID','DiseaseName','DiseaseID']], on='GeneID', how='inner')
pairs = pairs.drop_duplicates(['ChemicalID','GeneID','DiseaseID']).reset_index(drop=True)
print('Two-hop paths:', len(pairs))

MAX_EXAMPLES = 3000
pairs = pairs.sample(frac=1.0, random_state=42).reset_index(drop=True)
records = []
seen = set()
for _, r in pairs.iterrows():
    key = (str(r['ChemicalID']), str(r['GeneID']), str(r['DiseaseID']))
    if key in seen:
        continue
    seen.add(key)
    chemical = str(r['ChemicalName']).strip()
    gene = str(r['GeneSymbol']).strip()
    disease = str(r['DiseaseName']).strip()
    prompt = (
        f'Evidence 1: {chemical} has a CTD chemical-gene relationship with {gene}.\n'
        f'Evidence 2: the curated CTD gene-disease data links {gene} to {disease}.\n'
        f'Question: What disease is connected to {chemical} through gene {gene}? '        'Return Disease: <name> and Path: Chemical -> Gene -> Disease.'
    )
    answer = f'Disease: {disease}. Path: {chemical} -> {gene} -> {disease}.'
    records.append({'chemical': chemical, 'gene': gene, 'disease': disease, 'chemical_id': str(r['ChemicalID']), 'prompt': prompt, 'answer': answer})
    if len(records) >= MAX_EXAMPLES:
        break

data = pd.DataFrame(records)
chemicals = data['chemical_id'].drop_duplicates().sample(frac=1.0, random_state=42).tolist()
n_eval = max(1, int(len(chemicals) * 0.1))
eval_chems = set(chemicals[:n_eval])
train_df = data[~data['chemical_id'].isin(eval_chems)].copy()
eval_df = data[data['chemical_id'].isin(eval_chems)].head(150).copy()
print('Train:', len(train_df), 'Eval:', len(eval_df), 'Eval chemicals:', eval_df['chemical_id'].nunique())

Two-hop paths: 4861330
Train: 2678 Eval: 150 Eval chemicals: 94


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

def make_chat_text(prompt, answer):
    messages = [{'role':'user','content':prompt},{'role':'assistant','content':answer}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_ds = Dataset.from_dict({'text': [make_chat_text(p, a) for p, a in zip(train_df['prompt'], train_df['answer'])]})
print('SFT examples:', len(train_ds))

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).cuda()
model.config.use_cache = False

lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=['q_proj','k_proj','v_proj','o_proj'], bias='none', task_type='CAUSAL_LM')

sft_args = SFTConfig(
    output_dir='./outputs/ctd-standalone-sft',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    max_length=512,
    logging_steps=20,
    save_strategy='no',
    report_to='none',
    gradient_checkpointing=False,
    packing=False,
    dataset_text_field='text',
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
)

trainer = SFTTrainer(model=model, args=sft_args, train_dataset=train_ds, processing_class=tokenizer, peft_config=lora_config)
trainer.train()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/2678 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2678 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,2.179500
40,0.552400
60,0.320500
80,0.295500
100,0.296400
120,0.286000
140,0.279100
160,0.271800
180,0.284900
200,0.281200


TrainOutput(global_step=335, training_loss=0.41027074073677633, metrics={'train_runtime': 236.8961, 'train_samples_per_second': 11.305, 'train_steps_per_second': 1.414, 'total_flos': 885519034718208.0, 'train_loss': 0.41027074073677633})

In [5]:
def normalize(text):
    return re.sub(r'[^a-z0-9]+', ' ', str(text).lower()).strip()

def generate_batch(model, prompts, max_new_tokens=72, batch_size=8):
    outputs = []
    model.eval()
    for start in range(0, len(prompts), batch_size):
        batch = prompts[start:start+batch_size]
        chat_texts = [tokenizer.apply_chat_template([{'role':'user','content':p}], tokenize=False, add_generation_prompt=True) for p in batch]
        enc = tokenizer(chat_texts, return_tensors='pt', padding=True, truncation=True, max_length=384)
        enc = {k:v.to(model.device) for k,v in enc.items()}
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        prompt_len = enc['input_ids'].shape[1]
        outputs.extend(tokenizer.batch_decode(out[:, prompt_len:], skip_special_tokens=True))
    return outputs


## 1. Clean evaluation

This is the ordinary held-out Chemical -> Gene -> Disease task.

In [6]:
clean_preds = generate_batch(model, eval_df['prompt'].tolist())
clean_disease = []
clean_path = []
for (_, row), pred in zip(eval_df.iterrows(), clean_preds):
    p = normalize(pred)
    disease_ok = normalize(row['disease']) in p
    path_ok = disease_ok and normalize(row['gene']) in p
    clean_disease.append(disease_ok)
    clean_path.append(path_ok)
clean_metrics = {'disease_accuracy': sum(clean_disease)/len(clean_disease), 'chain_accuracy': sum(clean_path)/len(clean_path)}
print(clean_metrics)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{'disease_accuracy': 1.0, 'chain_accuracy': 1.0}


## 2. Distractor robustness

Three unrelated gene-disease edges are added. The model should still select the disease connected to the queried gene.

In [7]:
rng = random.Random(42)
pool = eval_df[['gene','disease']].drop_duplicates().to_dict('records')
d_rows = []
for _, row in eval_df.iterrows():
    candidates = [x for x in pool if x['gene'] != row['gene'] and x['disease'] != row['disease']]
    if len(candidates) < 3:
        continue
    distractors = rng.sample(candidates, 3)
    edges = [f"{row['gene']} -> {row['disease']}"] + [f"{x['gene']} -> {x['disease']}" for x in distractors]
    rng.shuffle(edges)
    prompt = f"Evidence: {row['chemical']} has a chemical-gene relationship with {row['gene']}.\nGene-disease evidence:\n- " + '\n- '.join(edges) + f"\nWhich disease is connected to {row['chemical']} through {row['gene']}? Return only the disease name."
    d_rows.append({**row.to_dict(), 'test_prompt': prompt})
d_df = pd.DataFrame(d_rows).head(150)
d_preds = generate_batch(model, d_df['test_prompt'].tolist())
d_hits = [normalize(r['disease']) in normalize(p) for (_, r), p in zip(d_df.iterrows(), d_preds)]
d_metrics = {'distractor_disease_accuracy': sum(d_hits)/len(d_hits)}
print(d_metrics)

{'distractor_disease_accuracy': 0.5266666666666666}


## 3. Counterfactual evidence following

The second hop is replaced by a synthetic disease. The model should follow the supplied hypothetical evidence and change its answer.

In [8]:
diseases = eval_df['disease'].drop_duplicates().tolist()
rng = random.Random(123)
cf_rows = []
for _, row in eval_df.iterrows():
    choices = [d for d in diseases if d != row['disease']]
    if not choices:
        continue
    cf = rng.choice(choices)
    prompt = (
        'Synthetic counterfactual test. Ignore prior knowledge and follow only the evidence below.\n'
        f"Evidence 1: {row['chemical']} is connected to gene {row['gene']}.\n"
        f"Evidence 2: Hypothetically, gene {row['gene']} is connected to {cf}.\n"
        f"Question: Which disease follows from this hypothetical evidence for {row['chemical']} through {row['gene']}? Return only the disease name."
    )
    cf_rows.append({**row.to_dict(), 'counterfactual_disease': cf, 'test_prompt': prompt})
cf_df = pd.DataFrame(cf_rows).head(150)
cf_preds = generate_batch(model, cf_df['test_prompt'].tolist())
follow = [normalize(r['counterfactual_disease']) in normalize(p) for (_, r), p in zip(cf_df.iterrows(), cf_preds)]
leak = [normalize(r['disease']) in normalize(p) for (_, r), p in zip(cf_df.iterrows(), cf_preds)]
cf_metrics = {'counterfactual_follow_accuracy': sum(follow)/len(follow), 'original_disease_leak_rate': sum(leak)/len(leak)}
print(cf_metrics)

{'counterfactual_follow_accuracy': 0.9666666666666667, 'original_disease_leak_rate': 0.0}


## 4. Summary

A stronger evidence-following model should keep high clean accuracy, remain robust to distractors, follow the synthetic counterfactual evidence, and have a low original-disease leak rate.

In [9]:
print('CTD REASONING DIAGNOSTICS')
print('-' * 72)
print(f"Clean disease accuracy         : {clean_metrics['disease_accuracy']:.3f}")
print(f"Clean chain accuracy           : {clean_metrics['chain_accuracy']:.3f}")
print(f"Distractor disease accuracy    : {d_metrics['distractor_disease_accuracy']:.3f}")
print(f"Counterfactual follow accuracy : {cf_metrics['counterfactual_follow_accuracy']:.3f}")
print(f"Original disease leak rate     : {cf_metrics['original_disease_leak_rate']:.3f}")

CTD REASONING DIAGNOSTICS
------------------------------------------------------------------------
Clean disease accuracy         : 1.000
Clean chain accuracy           : 1.000
Distractor disease accuracy    : 0.527
Counterfactual follow accuracy : 0.967
Original disease leak rate     : 0.000
